# Exploring a BraTS case

Interactive companion to `scripts/inspect_dataset.py`. It uses the same code as training, so what you
see here is what the model sees.

* Set the environment variable `BRATS_ROOT` (or edit `DATA_ROOT` below) to your local BraTS folder.
  Nothing is downloaded.
* This notebook is saved **without outputs**: every figure is produced from your own data when you run it.
* Research/educational use only.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
from IPython.display import Image, display

from brain_seg.config import load_config
from brain_seg.data import load_canonical_label, prepare_cases
from brain_seg.inference import summarize_label_map
from brain_seg.labels import CLASS_NAMES
from brain_seg.transforms import build_eval_transforms, build_train_transforms
from brain_seg.visualize import load_ras, plot_case_overview, reorient_to_ras

%matplotlib inline

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = os.environ.get("BRATS_ROOT", "/path/to/BraTS2021_Training_Data")
cfg = load_config(REPO / "configs" / "default.yaml")
cfg.data.num_workers = 0

## 1. Discover and validate cases
The label convention is detected from real mask values; incomplete cases raise an error.

In [ ]:
profile, cases = prepare_cases(cfg, data_root=DATA_ROOT)
print(f"{len(cases)} cases from {len({c.patient_id for c in cases})} patients, profile '{profile.name}'")
for raw, canonical in sorted(profile.label_map.items()):
    print(f"  raw label {raw} -> {CLASS_NAMES[canonical]:3s} {profile.label_descriptions.get(raw, '')}")

## 2. One case: four modalities and the reference segmentation
Volumes are reoriented to RAS; slices go through the largest tumour cross-section.

In [ ]:
case = cases[0]
affine = nib.load(str(case.label)).affine
print("orientation on disk:", "".join(nib.aff2axcodes(affine)),
      "| shape:", nib.load(str(case.label)).shape,
      "| spacing (mm):", tuple(float(z) for z in nib.load(str(case.label)).header.get_zooms()[:3]))
images = {m: load_ras(case.images[m]) for m in cfg.data.modalities}
label = reorient_to_ras(load_canonical_label(case.label, profile), affine)
out = plot_case_overview(images, label, REPO / "outputs" / "figures" / f"{case.case_id}_overview.png", case.case_id)
display(Image(filename=str(out)))

## 3. Tumour composition
Volumes per class and per BraTS region (ET ⊂ TC ⊂ WT).

In [ ]:
reference = load_canonical_label(case.label, profile)
spacing = tuple(float(s) for s in nib.load(str(case.label)).header.get_zooms()[:3])
summary = summarize_label_map(reference, spacing)
for name, stats in {**summary["classes"], **summary["regions"]}.items():
    print(f"{name:3s} {stats['volume_ml']:8.2f} mL")
brain_voxels = int((images["flair"] > 0).sum())
print(f"whole tumour = {100 * summary['regions']['WT']['voxels'] / brain_voxels:.1f}% of brain voxels")

## 4. Raw intensity distributions
MRI intensities have no absolute unit and vary between scanners, which is why each modality is z-scored inside the brain.

In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(3.2 * len(images), 2.6))
for ax, (name, volume) in zip(axes, images.items()):
    ax.hist(volume[volume > 0].ravel(), bins=100, color="#2a78d6")
    ax.set_title(name.upper())
    ax.set_yticks([])
fig.tight_layout()
plt.show()

## 5. What the network receives
Full-volume preprocessing (used for validation and inference) and one training patch.

In [ ]:
item = case.to_monai(cfg.data.modalities)
full = build_eval_transforms(cfg, profile, with_label=True)(item)
print("preprocessed volume:", tuple(full["image"].shape), "label:", tuple(full["label"].shape))
brain = full["image"][0] != 0
print("channel means/stds inside brain:",
      [(round(float(c[brain].mean()), 3), round(float(c[brain].std()), 3)) for c in full["image"]])

patch = build_train_transforms(cfg, profile)(item)[0]
values, counts = torch.unique(patch["label"], return_counts=True)
print("training patch:", tuple(patch["image"].shape),
      {CLASS_NAMES[int(v)]: f"{100 * int(c) / patch['label'].numel():.1f}%" for v, c in zip(values, counts)})